In [1]:
"""
Interactive Calgary map:
- Calgary bus stops (official Open Calgary)
- Calgary LRT stations (official Open Calgary)
- Grocery stores (OpenStreetMap via OSMnx)
- Shows grocery stores within 500 m of transit
- Legend uses the exact same icons as the map markers
- Includes reset-view button
- Saves output to HTML
"""

import warnings
warnings.filterwarnings("ignore")

import urllib.parse
import html

import pandas as pd
import geopandas as gpd
import requests
import folium
import osmnx as ox
from branca.element import Element

CITY_NAME = "Calgary, Alberta, Canada"
BUFFER_METERS = 500
OUTPUT_HTML = "calgary_grocery_transit_500m_map_fixed.html"
DEFAULT_CENTER = [51.0447, -114.0719]
DEFAULT_ZOOM = 11

BUS_STOPS_URL = "https://data.calgary.ca/resource/muzh-c9qc.geojson?$limit=50000"
LRT_STATIONS_URL = "https://data.calgary.ca/resource/2axz-xm4q.geojson?$limit=5000"

PROJECTED_CRS = "EPSG:32611"
WGS84 = "EPSG:4326"


def load_geojson_as_gdf(url: str) -> gpd.GeoDataFrame:
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    return gpd.GeoDataFrame.from_features(response.json()["features"], crs=WGS84)


def clean_bus_stops(bus_stops: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    cols = [c for c in ["stop_name", "stop_id", "status", "geometry"] if c in bus_stops.columns]
    gdf = bus_stops[cols].copy()
    if "status" in gdf.columns:
        gdf = gdf[gdf["status"].fillna("").str.upper() == "ACTIVE"]
    gdf["type"] = "Bus stop"
    gdf["name"] = gdf["stop_name"] if "stop_name" in gdf.columns else "Bus stop"
    return gdf


def clean_lrt_stations(lrt: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    name_col = next((c for c in ["station_name", "name", "description", "station"] if c in lrt.columns), None)
    cols = [c for c in [name_col, "geometry"] if c]
    gdf = lrt[cols].copy()
    gdf["type"] = "LRT station"
    gdf["name"] = gdf[name_col] if name_col else "LRT station"
    return gdf


def load_grocery_stores(city_name: str) -> gpd.GeoDataFrame:
    tags = {"shop": ["supermarket", "grocery", "convenience"]}
    gdf = ox.features_from_place(city_name, tags=tags).reset_index()
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf.to_crs(PROJECTED_CRS)
    gdf["geometry"] = gdf.geometry.centroid
    gdf = gdf.to_crs(WGS84)
    gdf["name"] = gdf["name"].fillna("Unnamed grocery store") if "name" in gdf.columns else "Unnamed grocery store"
    gdf["type"] = "Grocery store"
    keep_cols = [c for c in ["name", "shop", "brand", "addr:full", "type", "geometry"] if c in gdf.columns]
    return gdf[keep_cols].drop_duplicates(subset=["name", "geometry"]).copy()


def popup_html_store(row):
    bits = [f"<b>{html.escape(str(row.get('name', 'Grocery store')))}</b>"]
    if pd.notna(row.get("brand")):
        bits.append(f"Brand: {html.escape(str(row.get('brand')))}")
    if pd.notna(row.get("shop")):
        bits.append(f"Shop type: {html.escape(str(row.get('shop')))}")
    if pd.notna(row.get("addr:full")):
        bits.append(f"Address: {html.escape(str(row.get('addr:full')))}")
    return "<br>".join(bits)


def popup_html_transit(row):
    bits = [f"<b>{html.escape(str(row.get('name', 'Transit point')))}</b>"]
    if pd.notna(row.get("stop_id")):
        bits.append(f"Stop ID: {html.escape(str(row.get('stop_id')))}")
    bits.append(f"Type: {html.escape(str(row.get('type', 'Transit')))}")
    return "<br>".join(bits)


def pin_icon_data_uri(color_hex: str, glyph: str) -> str:
    svg = f"""
    <svg xmlns="http://www.w3.org/2000/svg" width="44" height="57" viewBox="0 0 44 57">
      <defs>
        <filter id="shadow" x="-20%" y="-20%" width="140%" height="140%">
          <feDropShadow dx="1" dy="2" stdDeviation="2" flood-color="#888" flood-opacity="0.35"/>
        </filter>
      </defs>
      <path filter="url(#shadow)" d="M22 2C10.95 2 2 10.95 2 22c0 14.5 20 33 20 33s20-18.5 20-33C42 10.95 33.05 2 22 2z" fill="{color_hex}"/>
      <circle cx="22" cy="18" r="11.5" fill="white" fill-opacity="0.22"/>
      <text x="22" y="22" text-anchor="middle" font-size="16" font-family="Arial, sans-serif">{glyph}</text>
    </svg>
    """
    return "data:image/svg+xml;charset=utf-8," + urllib.parse.quote(svg)


bus_stops = clean_bus_stops(load_geojson_as_gdf(BUS_STOPS_URL))
lrt_stations = clean_lrt_stations(load_geojson_as_gdf(LRT_STATIONS_URL))
groceries = load_grocery_stores(CITY_NAME)

transit_gdf = gpd.GeoDataFrame(pd.concat([bus_stops, lrt_stations], ignore_index=True), geometry="geometry", crs=WGS84)
transit_proj = transit_gdf.to_crs(PROJECTED_CRS)
groceries_proj = groceries.to_crs(PROJECTED_CRS)

transit_buffers = transit_proj.copy()
transit_buffers["geometry"] = transit_buffers.geometry.buffer(BUFFER_METERS)

near_join = gpd.sjoin(
    groceries_proj,
    transit_buffers[["geometry"]],
    predicate="within",
    how="inner"
).drop(columns=["index_right"], errors="ignore").drop_duplicates(subset=["name", "geometry"])

transit_web = transit_gdf.to_crs(WGS84)
groceries_web = groceries.to_crs(WGS84)
nearby_web = near_join.to_crs(WGS84)
buffer_union_web = gpd.GeoSeries(transit_buffers.union_all(), crs=PROJECTED_CRS).to_crs(WGS84)

m = folium.Map(location=DEFAULT_CENTER, zoom_start=DEFAULT_ZOOM, tiles="CartoDB positron")

bus_icon_url = pin_icon_data_uri("#3B82F6", "")
train_icon_url = pin_icon_data_uri("#E74C3C", "")
grocery_icon_url = pin_icon_data_uri("#7CB342", "")

bus_icon = folium.CustomIcon(bus_icon_url, icon_size=(34, 44), icon_anchor=(17, 42), popup_anchor=(0, -34))
train_icon = folium.CustomIcon(train_icon_url, icon_size=(34, 44), icon_anchor=(17, 42), popup_anchor=(0, -34))
grocery_icon = folium.CustomIcon(grocery_icon_url, icon_size=(34, 44), icon_anchor=(17, 42), popup_anchor=(0, -34))

folium.GeoJson(
    buffer_union_web.__geo_interface__,
    name="500m transit coverage",
    style_function=lambda x: {"fillColor": "#f6c344", "color": "#c99a00", "weight": 1, "fillOpacity": 0.10},
    tooltip="Combined 500 m transit coverage"
).add_to(m)

bus_fg = folium.FeatureGroup(name="Bus stops", show=True)
lrt_fg = folium.FeatureGroup(name="LRT stations", show=True)
all_grocery_fg = folium.FeatureGroup(name="All grocery stores", show=False)
near_grocery_fg = folium.FeatureGroup(name=f"Grocery stores within {BUFFER_METERS}m", show=True)

for _, row in transit_web[transit_web["type"] == "Bus stop"].iterrows():
    folium.Marker([row.geometry.y, row.geometry.x], popup=folium.Popup(popup_html_transit(row), max_width=280), tooltip=row.get("name", "Bus stop"), icon=bus_icon).add_to(bus_fg)

for _, row in transit_web[transit_web["type"] == "LRT station"].iterrows():
    folium.Marker([row.geometry.y, row.geometry.x], popup=folium.Popup(popup_html_transit(row), max_width=280), tooltip=row.get("name", "LRT station"), icon=train_icon).add_to(lrt_fg)

for _, row in groceries_web.iterrows():
    folium.Marker([row.geometry.y, row.geometry.x], popup=folium.Popup(popup_html_store(row), max_width=280), tooltip=row.get("name", "Grocery store"), icon=grocery_icon).add_to(all_grocery_fg)

for _, row in nearby_web.iterrows():
    folium.Marker([row.geometry.y, row.geometry.x], popup=folium.Popup(popup_html_store(row), max_width=280), tooltip=row.get("name", "Grocery store within 500m"), icon=grocery_icon).add_to(near_grocery_fg)

for fg in [bus_fg, lrt_fg, all_grocery_fg, near_grocery_fg]:
    fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

title_html = f"""
<div style="position: fixed; top: 14px; left: 50px; z-index: 9999; background: white; padding: 12px 14px; border: 1px solid #d9d9d9; border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.15); font-family: Arial, sans-serif; font-size: 14px; line-height: 1.4;">
  <div style="font-size:16px; font-weight:bold; margin-bottom:6px;">Calgary groceries near transit</div>
  <div>Groceries within <b>{BUFFER_METERS} m</b> of Calgary transit.</div>
</div>
"""
m.get_root().html.add_child(Element(title_html))

legend_html = f"""
<div style="position: fixed; bottom: 30px; left: 50px; z-index: 9999; background: white; padding: 12px 14px; border: 1px solid #d9d9d9; border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.15); font-family: Arial, sans-serif; font-size: 13px; min-width: 280px; line-height: 1.5;">
  <div style="font-size:14px; font-weight:bold; margin-bottom:8px;">Legend</div>
  <div style="display:flex; align-items:center; margin-bottom:6px;"><img src="{bus_icon_url}" style="width:20px; height:auto; margin-right:8px;" alt="Bus stop icon"><span>Bus stop</span></div>
  <div style="display:flex; align-items:center; margin-bottom:6px;"><img src="{train_icon_url}" style="width:20px; height:auto; margin-right:8px;" alt="LRT station icon"><span>LRT station</span></div>
  <div style="display:flex; align-items:center; margin-bottom:6px;"><img src="{grocery_icon_url}" style="width:20px; height:auto; margin-right:8px;" alt="Grocery store icon"><span>Grocery store</span></div>
  <div style="display:flex; align-items:center;"><span style="display:inline-block; width:18px; height:12px; background:rgba(246,195,68,0.35); border:1px solid #c99a00; margin-right:8px;"></span><span>500 m transit coverage area</span></div>
</div>
"""
m.get_root().html.add_child(Element(legend_html))

map_var = m.get_name()
reset_js = f"""
<script>
function resetMapView() {{
    {map_var}.closePopup();
    {map_var}.setView([{DEFAULT_CENTER[0]}, {DEFAULT_CENTER[1]}], {DEFAULT_ZOOM});
}}
</script>
"""
m.get_root().html.add_child(Element(reset_js))

reset_button_html = """
<div style="position: fixed; top: 120px; left: 50px; z-index: 9999;">
  <button type="button" onclick="resetMapView()" style="background: white; border: 1px solid #d9d9d9; border-radius: 8px; padding: 10px 12px; box-shadow: 0 2px 8px rgba(0,0,0,0.15); font-family: Arial, sans-serif; font-size: 13px; cursor: pointer;">Reset view</button>
</div>
"""
m.get_root().html.add_child(Element(reset_button_html))

m.save(OUTPUT_HTML)
print(f"Saved interactive map to: {OUTPUT_HTML}")


Saved interactive map to: calgary_grocery_transit_500m_map_fixed.html
